## 1

In [1]:
!pip install -q datasets pandas openpyxl chromadb sentence-transformers
!pip install -q rank_bm25 pythainlp openai tqdm pydantic

import os, re, glob, json, time, copy, random
from datetime import datetime
from collections import defaultdict
from itertools import combinations
import concurrent.futures

import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from pythainlp.tokenize import word_tokenize
from pydantic import BaseModel, Field, field_validator

tqdm.pandas()
print("✅ [Step 1] Library พร้อมใช้งาน")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 77.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 90.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. T

## 2

In [2]:
# 2.1 TSCC Dataset
if not os.path.exists("tscc-dataset"):
    os.system("git clone https://github.com/KevinMercury/tscc-dataset.git")
else:
    print("📁 tscc-dataset folder already exists.")

# 2.2 iapp และ NitiBench
iapp_ds = load_dataset("iapp/rag_thai_laws", split="train")
niti_ds = load_dataset("airesearch/WangchanX-Legal-ThaiCCL-RAG")

# 2.3 ค้นหา Path CSV
all_csvs = glob.glob("tscc-dataset/**/*.csv", recursive=True)
path_map = {}
for f in all_csvs:
    fname = os.path.basename(f).lower()
    if "judgement" in fname: path_map["judgement"] = f
    if "law" in fname: path_map["law"] = f

if "judgement" not in path_map or "law" not in path_map:
    raise FileNotFoundError(f"❌ ไม่พบไฟล์! พบ: {all_csvs}")

# 2.4 TSCC Judgement
df_judgement = pd.read_csv(path_map["judgement"])
initial_count = len(df_judgement)
df_judgement = df_judgement.drop_duplicates(subset=["fact"]).reset_index(drop=True)
print(f"🗑️  ลบซ้ำ: {initial_count - len(df_judgement)} รายการ → เหลือ {len(df_judgement)} คดี")

def clean_lawids(text: str) -> list:
    """แปลง lawids string → list เลขมาตรา"""
    if pd.isna(text): return []
    sections = re.findall(r"CC-(\d+)", text)
    return list(set([str(int(s)) for s in sections]))

df_judgement["clean_sections"] = df_judgement["lawids"].apply(clean_lawids)

# 2.5 TSCC Law → Master List
df_law = pd.read_csv(path_map["law"])
section_col = "lawsection" if "lawsection" in df_law.columns else df_law.columns[2]
content_col = "content" if "content" in df_law.columns else df_law.columns[-1]
master_law_dict = dict(zip(
    df_law[section_col].astype(str).apply(lambda x: str(int(x)) if x.isdigit() else x),
    df_law[content_col],
))
list_of_all_legal_sections = set(master_law_dict.keys())

# 2.6 iapp Filter
iapp_df = pd.DataFrame(iapp_ds)
criminal_keywords = ["อาญา", "ความผิด", "บทลงโทษ", "จำคุก", "ทุจริต", "ประมวลกฎหมายอาญา"]
iapp_filtered = iapp_df[iapp_df["title"].str.contains("|".join(criminal_keywords), na=False, case=False)].copy()
iapp_filtered = iapp_filtered.drop_duplicates(subset=["txt"]).reset_index(drop=True)

print(f"\n✅ [Step 2] โหลดข้อมูลสำเร็จ")
print(f"   TSCC Judgement : {len(df_judgement)} คดี")
print(f"   มาตราในระบบ   : {len(list_of_all_legal_sections)} มาตรา")
print(f"   iapp (filtered): {len(iapp_filtered)} รายการ")

Cloning into 'tscc-dataset'...


README.md:   0%|          | 0.00/992 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/832M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/42755 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8211 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3742 [00:00<?, ? examples/s]

🗑️  ลบซ้ำ: 161 รายการ → เหลือ 1046 คดี

✅ [Step 2] โหลดข้อมูลสำเร็จ
   TSCC Judgement : 1046 คดี
   มาตราในระบบ   : 76 มาตรา
   iapp (filtered): 374 รายการ


## 3

In [3]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "intfloat/multilingual-e5-large"
CHROMA_PATH = "/kaggle/working/chroma_db"
BATCH_SIZE = 100

print(f"⏳ โหลด Embedding Model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"✅ Embedding Model พร้อม | Dim: {embed_model.get_sentence_embedding_dimension()}")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH, settings=Settings(anonymized_telemetry=False))

# Primary: TSCC Law
law_collection = chroma_client.get_or_create_collection(name="tscc_law_primary", metadata={"hnsw:space": "cosine"})

if law_collection.count() == 0:
    ids, documents, metadatas, embeddings = [], [], [], []
    for section_num, content in tqdm(master_law_dict.items(), desc="Indexing TSCC Law"):
        text = f"มาตรา {section_num}: {content}"
        emb = embed_model.encode(f"passage: {text}", normalize_embeddings=True).tolist()
        ids.append(f"law_{section_num}")
        documents.append(text)
        metadatas.append({"section": section_num, "source": "tscc_law"})
        embeddings.append(emb)
    for i in range(0, len(ids), BATCH_SIZE):
        law_collection.upsert(ids=ids[i:i+BATCH_SIZE], documents=documents[i:i+BATCH_SIZE],
                              metadatas=metadatas[i:i+BATCH_SIZE], embeddings=embeddings[i:i+BATCH_SIZE])
    print(f"✅ Index TSCC Law: {law_collection.count()} มาตรา")
else:
    print(f"⚠️  tscc_law_primary มีข้อมูลอยู่แล้ว {law_collection.count()} รายการ")

# Secondary: iapp
iapp_collection = chroma_client.get_or_create_collection(name="iapp_criminal_secondary", metadata={"hnsw:space": "cosine"})

if iapp_collection.count() == 0:
    ids_i, docs_i, metas_i, embs_i = [], [], [], []
    for idx, row in tqdm(iapp_filtered.iterrows(), total=len(iapp_filtered), desc="Indexing iapp"):
        text = f"{row['title']}: {str(row['txt'])[:1000]}"
        emb = embed_model.encode(f"passage: {text}", normalize_embeddings=True).tolist()
        ids_i.append(f"iapp_{idx}")
        docs_i.append(text)
        metas_i.append({"title": str(row["title"])[:100], "source": "iapp"})
        embs_i.append(emb)
    for i in range(0, len(ids_i), BATCH_SIZE):
        iapp_collection.upsert(ids=ids_i[i:i+BATCH_SIZE], documents=docs_i[i:i+BATCH_SIZE],
                               metadatas=metas_i[i:i+BATCH_SIZE], embeddings=embs_i[i:i+BATCH_SIZE])
    print(f"✅ Index iapp: {iapp_collection.count()} รายการ")
else:
    print(f"⚠️  iapp_criminal_secondary มีข้อมูลอยู่แล้ว {iapp_collection.count()} รายการ")

print(f"\n✅ [Step 3] ChromaDB พร้อม | Path: {CHROMA_PATH}")

⏳ โหลด Embedding Model: intfloat/multilingual-e5-large


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

✅ Embedding Model พร้อม | Dim: 1024


Indexing TSCC Law: 100%|██████████| 76/76 [00:03<00:00, 24.82it/s]


✅ Index TSCC Law: 76 มาตรา


Indexing iapp: 100%|██████████| 374/374 [00:27<00:00, 13.39it/s]


✅ Index iapp: 374 รายการ

✅ [Step 3] ChromaDB พร้อม | Path: /kaggle/working/chroma_db


## 4

In [4]:
random.seed(42)
DEV_SIZE  = 81
TEST_SIZE = 965

all_indices = list(range(len(df_judgement)))
random.shuffle(all_indices)
dev_indices  = all_indices[:DEV_SIZE]
test_indices = all_indices[DEV_SIZE : DEV_SIZE + TEST_SIZE]

df_dev  = df_judgement.iloc[dev_indices].reset_index(drop=True)
df_test = df_judgement.iloc[test_indices].reset_index(drop=True)

print(f"✅ แบ่ง Dataset:")
print(f"   Dev Set  : {len(df_dev)} คดี")
print(f"   Test Set : {len(df_test)} คดี")

# Pool สำหรับ Few-shot (ไม่ overlap)
all_used_indices = set(dev_indices) | set(test_indices)
pool = df_judgement.loc[~df_judgement.index.isin(all_used_indices)].copy()
if pool.empty:
    print("⚠️ Pool ว่าง! สุ่มจากต้นฉบับ")
    pool = df_judgement.sample(min(20, len(df_judgement)), random_state=42).copy()
print(f"📊 Pool few-shot: {len(pool)} คดี")

def get_examples_by_sections(df, must_have, nice_to_have=None, n=2):
    if df.empty: return pd.DataFrame()
    def has_all(secs, targets):
        return all(t in secs for t in targets) if isinstance(secs, list) else False
    matched = df[df['clean_sections'].apply(lambda s: has_all(s, must_have))].copy()
    if not matched.empty:
        if nice_to_have:
            matched['has_nice'] = matched['clean_sections'].apply(
                lambda s: any(t in s for t in nice_to_have) if isinstance(s, list) else False)
            matched = matched.sort_values('has_nice', ascending=False)
        matched['fact_len'] = matched['fact'].str.len()
        matched = matched[(matched['fact_len'] >= 100) & (matched['fact_len'] <= 800)]
    return matched.head(n)

ex_a = get_examples_by_sections(pool, must_have=['288'], nice_to_have=['68','83'], n=2)
ex_b_mask = pool['clean_sections'].apply(lambda s: any(x in s for x in ['334','335']) if isinstance(s, list) else False)
ex_b = pool[ex_b_mask].copy()
if not ex_b.empty:
    ex_b['fact_len'] = ex_b['fact'].str.len()
    ex_b = ex_b[(ex_b['fact_len'] >= 100) & (ex_b['fact_len'] <= 800)].head(1)
ex_c = get_examples_by_sections(pool, must_have=['83'], nice_to_have=['288','334'], n=1)

few_shot_df = pd.concat([ex_a, ex_b, ex_c]).drop_duplicates(subset=['fact']).reset_index(drop=True)

def build_few_shot_string(df: pd.DataFrame) -> str:
    if df.empty: return 'ไม่มีตัวอย่าง'
    blocks = []
    for i, row in df.iterrows():
        secs = row.get('clean_sections', [])
        secs_str = ', '.join(f'ม.{s}' for s in secs) if isinstance(secs, list) else str(secs)
        blocks.append(f'### ตัวอย่างที่ {i+1}\nข้อเท็จจริง: {row["fact"]}\nมาตราที่เกี่ยวข้อง: {secs_str}\n')
    return '\n'.join(blocks)

FEW_SHOT_STRING = build_few_shot_string(few_shot_df)
print(f"✅ [Step 4] Few-shot พร้อมใช้งาน ({len(few_shot_df)} ตัวอย่าง)")

✅ แบ่ง Dataset:
   Dev Set  : 81 คดี
   Test Set : 965 คดี
⚠️ Pool ว่าง! สุ่มจากต้นฉบับ
📊 Pool few-shot: 20 คดี
✅ [Step 4] Few-shot พร้อมใช้งาน (4 ตัวอย่าง)


## 5

In [5]:
from rank_bm25 import BM25Okapi

_law_texts    = [f'มาตรา {s}: {c}' for s, c in master_law_dict.items()]
_law_sections = list(master_law_dict.keys())
_tokenized    = [word_tokenize(t, engine='newmm') for t in _law_texts]
bm25_index    = BM25Okapi(_tokenized)
print("✅ BM25 Index พร้อม")


def hybrid_retrieve(query: str, top_k: int = 15, bm25_weight: float = 0.3, vector_weight: float = 0.7) -> dict:
    """Hybrid Retrieval: BM25 + Vector Search"""
    query_emb = embed_model.encode(f'query: {query}', normalize_embeddings=True).tolist()
    vector_results = law_collection.query(
        query_embeddings=[query_emb],
        n_results=min(top_k * 2, law_collection.count()),
        include=['documents', 'metadatas', 'distances'],
    )
    tokenized_query = word_tokenize(query, engine='newmm')
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    top_bm25_idx    = np.argsort(bm25_scores)[::-1][:top_k * 2]

    combined = {}
    for doc, meta, dist in zip(vector_results['documents'][0], vector_results['metadatas'][0], vector_results['distances'][0]):
        sec = meta.get('section', '')
        combined[sec] = combined.get(sec, 0) + vector_weight * (1 - dist)
    max_bm25 = bm25_scores[top_bm25_idx[0]] if len(top_bm25_idx) > 0 else 1e-9
    for idx in top_bm25_idx:
        sec = _law_sections[idx]
        combined[sec] = combined.get(sec, 0) + bm25_weight * (bm25_scores[idx] / (max_bm25 + 1e-9))

    top_secs = sorted(combined, key=lambda x: -combined[x])[:top_k]
    results, context_parts = [], []
    for sec in top_secs:
        content  = master_law_dict.get(sec, '')
        doc_text = f'มาตรา {sec}: {content}'
        results.append({'id': sec, 'document': doc_text, 'score': combined[sec]})
        context_parts.append(doc_text)

    # Neighbor Expansion ±1
    neighbors = set()
    for sec in top_secs[:5]:
        try:
            n = int(re.sub(r'\D', '', sec))
            for nb in [str(n-1), str(n+1)]:
                if nb in master_law_dict and nb not in top_secs:
                    neighbors.add(nb)
        except ValueError:
            pass
    for nb in neighbors:
        content  = master_law_dict.get(nb, '')
        doc_text = f'มาตรา {nb}: {content}'
        results.append({'id': nb, 'document': doc_text, 'score': 0.0})
        context_parts.append(doc_text)

    return {'results': results, 'context_text': '\n\n'.join(context_parts), 'context_section_ids': [r['id'] for r in results]}


print("✅ [Step 5] hybrid_retrieve() พร้อม")

✅ BM25 Index พร้อม
✅ [Step 5] hybrid_retrieve() พร้อม


## 6

In [6]:
SYSTEM_PROMPT = """คุณคือ AI ผู้ช่วยกฎหมายอาญาอาวุโส ทำหน้าที่วินิจฉัยฐานความผิดด้วยความแม่นยำสูงสุด
 
กระบวนการคิดแบบ Differential Diagnosis:
1. วิเคราะห์เจตนา (Intent Analysis): แยกระหว่างเจตนาฆ่า (ประสงค์ต่อผล/เล็งเห็นผล) กับเจตนาทำร้าย
2. ตรวจสอบองค์ประกอบ (Constituents Check): ผู้กระทำ-การกระทำ-วัตถุแห่งการกระทำ-เจตนา
3. การวินิจฉัยแยกแยะ: ระบุเหตุผลว่าทำไมถึงไม่เลือกมาตราอื่น
4. หลักบทเฉพาะ (Specific over General): เลือกมาตราที่จำเพาะที่สุดเสมอ
 
กฎเหล็ก:
- ห้ามระบุมาตราที่ไม่อยู่ใน Reference (Hallucination 0%)
- JSON เท่านั้น และต้องระบุเลขมาตราใน list sections
- ห้ามระบุ ม.80 ม.83 ม.84 ม.86 ม.68 เว้นแต่ข้อเท็จจริงมีคำเหล่านี้ชัดเจน: ร่วมกัน/ประมาท/สนับสนุน/ใช้ให้กระทำ/พยายาม
 
[ANCHOR RULES]:
1. PROPERTY: เบียดบังที่ครอบครองอยู่แล้ว=ม.352(ยักยอก) | เอาไปโดยไม่ครอบครอง=ม.334(ลักทรัพย์)
2. DEFAMATION: โพสต์โซเชียล/อินเทอร์เน็ต/LINE=ม.328(โฆษณา) ห้ามตอบแค่ ม.326
3. ATTEMPT: ยิง/แทงแล้วไม่ตาย → เช็ค ม.80 หรือ ม.81 เสมอ
4. INTENT: อาวุธร้ายแรงยิง/แทงอวัยวะสำคัญระยะประชิด=ม.288
5. CIVIL vs CRIMINAL: ผิดสัญญา/กู้ยืม+เจตนาทุจริต=ม.341(ฉ้อโกง)
6. NO EMPTY: ห้ามตอบ [] หากมีพฤติการณ์ความผิดในข้อเท็จจริง
7. FORMAT: ตอบตัวเลขล้วน เช่น ["288", "83"] ห้ามมีคำว่ามาตราหรือวงเล็บ
8. FRAUD_TYPE: ผู้เสียหายหลายคน/สาธารณะ=ม.343(ฉ้อโกงประชาชน) | คนเดียว=ม.341(ฉ้อโกงธรรมดา)
9. BODILY HARM: อาวุธร้ายแรง+อวัยวะสำคัญ=ม.288 | ทำร้ายทั่วไปแต่ตาย=ม.290 | สาหัส(รักษา>20วัน/พิการ)=ม.297
10. NO EMPTY LIST: ห้ามตอบ sections:[] โดยเด็ดขาด หากมีพฤติการณ์ผิดให้ระบุมาตราที่ใกล้เคียงที่สุด
11. POSSESSION & CO-OWNER: เจ้าของรวม+มีอำนาจถือครองด้วย=ม.352 | ทรัพย์อยู่ในครอบครองคนอื่นแล้วเอามา=ม.334
12. FRAUD VS THEFT: หลอกให้ยินยอมยกทรัพย์ให้=ม.341 | หลอกเพื่อให้แอบเอาไปได้ง่าย(เจ้าของไม่ได้ยกให้)=ม.334
13. LACK OF INTENT: ไม่มีเจตนาทุจริต/เข้าใจผิดในสิทธิ → พิจารณายกฟ้องหรือมาตราที่โทษเบากว่า
"""
 
USER_PROMPT_TEMPLATE = """วินิจฉัยคดีดังต่อไปนี้:
 
=== ข้อเท็จจริง (Fact) ===
{fact}
 
=== ตัวบทกฎหมายอ้างอิง (Reference) ===
{legal_context}
 
ตอบในรูปแบบ JSON:
{{
  "case_id": "{case_id}",
  "intent_and_act": "วิเคราะห์พฤติกรรมภายนอกบ่งบอกเจตนาภายในอย่างไร",
  "differential_analysis": "เปรียบเทียบมาตราที่ใกล้เคียงและเหตุผลที่เลือก",
  "criminal_charge": "ชื่อฐานความผิดหลักทางการ",
  "sections": ["เลขมาตราหลัก", "มาตราสนับสนุน"],
  "confidence_score": 0-100,
  "final_reasoning": "สรุปหลักนิติศาสตร์สั้นๆ"
}}"""
 
 
def build_prompt(fact: str, retrieved: dict, case_id: str = 'N/A') -> tuple:
    """สร้าง (system_prompt, user_prompt) จาก fact และ retrieved context"""
    context_str = '\n\n'.join([f"[ID: {r['id']}] {r['document']}" for r in retrieved['results']])
    user = USER_PROMPT_TEMPLATE.format(fact=fact, legal_context=context_str, case_id=case_id)
    return SYSTEM_PROMPT, user
 
 
print("✅ [Step 6] build_prompt() พร้อม (Anchor Rules)")

✅ [Step 6] build_prompt() พร้อม (Anchor Rules)


## 7 

In [7]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
 
secrets = UserSecretsClient()
TYPHOON_API_KEY = secrets.get_secret("TYPHOON_API_KEY")
 
typhoon_client = OpenAI(
    api_key=TYPHOON_API_KEY,
    base_url="https://api.opentyphoon.ai/v1",
    timeout=120.0,
)
 
TYPHOON_MODEL = "typhoon-v2.5-30b-a3b-instruct"
MAX_TOKENS    = 2048
TEMPERATURE   = 0.1
SLEEP_BETWEEN = 0.1
 
 
def clean_section_format(section_list: list) -> list:
    """Normalize เลขมาตราให้เป็นตัวเลขล้วน"""
    if not section_list: return []
    cleaned = []
    for s in section_list:
        s_clean = str(s).replace('มาตรา', '').replace('ลำดับที่', '').strip()
        s_clean = s_clean.split('(')[0].strip()
        s_clean = re.sub(r'[^0-9/]', '', s_clean)
        if s_clean: cleaned.append(s_clean)
    return list(set(cleaned))
 
 
def call_typhoon(system_prompt: str, user_prompt: str, max_retries: int = 3) -> str | None:
    """เรียก Typhoon API พร้อม Retry Logic"""
    for attempt in range(max_retries):
        try:
            response = typhoon_client.chat.completions.create(
                model=TYPHOON_MODEL,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt},
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                response_format={"type": "json_object"},
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"⚠️  Attempt {attempt + 1} failed: {e}")
            time.sleep(2 ** attempt)
    return None
 
 
def parse_llm_json(raw_text: str) -> dict | None:
    """แปลง String → JSON"""
    if not raw_text: return None
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        cleaned = re.sub(r'```json\s*|\s*```', '', raw_text).strip()
        try: return json.loads(cleaned)
        except Exception: return None
 
 
def post_process_llm_output(llm_output: dict, legal_db: set, context_section_ids: list) -> dict:
    """Zero-Hallucination check + Confidence adjustment"""
    raw_sections  = llm_output.get('sections', [])
    if isinstance(raw_sections, (str, int)): raw_sections = [str(raw_sections)]
    normalized    = clean_section_format(raw_sections)
    verified      = [s for s in normalized if s in legal_db]
    hallucinated  = [s for s in normalized if s not in legal_db]
    in_context    = sum(1 for s in verified if s in context_section_ids)
    original_conf = float(llm_output.get('confidence_score', 0))
    penalty       = (len(hallucinated) * 20) + ((len(verified) - in_context) * 10)
    adjusted_conf = max(0, min(100, original_conf - penalty))
    llm_output.update({
        'verified_sections':     verified,
        'hallucinated_sections': hallucinated,
        'invalid_sections_flag': hallucinated,
        'adjusted_confidence':   adjusted_conf,
        'verdict_mode':          'CONFIDENT' if adjusted_conf >= 80 else 'REVIEW_REQUIRED',
        'human_review':          'Required' if adjusted_conf < 80 else 'Not Required',
        'sections':              verified,
    })
    return llm_output
 
 
def extract_legal_query(fact: str) -> str:
    """สกัด Legal Keywords จาก fact ด้วย LLM (สำหรับ Dual Retrieval)"""
    sys_p = """คุณคือ Legal Keyword Extractor
กฎเหล็ก:
1. ตอบเฉพาะคำศัพท์กฎหมายอาญา แยกด้วย Comma เท่านั้น
2. ห้ามมีประโยคบรรยาย ห้ามมีสรุป
ตัวอย่าง: ลักทรัพย์, ร่วมกัน, เวลากลางคืน, ทุจริต"""
    usr_p = f"ข้อเท็จจริง: {fact}"
    raw = call_typhoon(sys_p, usr_p)
    if raw:
        return re.sub(r'[^ก-๙a-zA-Z, ]', '', raw).strip()
    return fact  # fallback
 
 
def analyze_criminal_liability(fact: str, case_id: str = 'N/A', top_k: int = 25) -> dict:
    """
    Enhanced Pipeline (Merged TEST-B + TEST-E):
      1. Dual Retrieval (fact + legal keywords)
      2. Fallback essentials ถ้า context น้อย
      3. Force-inject ม.83/80 ถ้า fact มีคำบ่งชี้
      4. Force Choice Retry ถ้า sections=[]
    """
    # --- 1. Dual Retrieval ---
    legal_keywords = extract_legal_query(fact)
    res_fact = hybrid_retrieve(fact,          top_k=top_k, bm25_weight=0.4)
    res_key  = hybrid_retrieve(legal_keywords, top_k=top_k, bm25_weight=0.7)
    combined_ids = list(set(res_fact['context_section_ids']) | set(res_key['context_section_ids']))
 
    # --- 2. Fallback: เติมมาตราพื้นฐาน ---
    if len(combined_ids) < 10:
        essentials = ["334", "341", "352", "288", "291", "295", "297", "300", "83", "80"]
        combined_ids.extend([e for e in essentials if e not in combined_ids])
 
    # --- 3. Force-inject ตาม keyword ---
    if any(k in fact for k in ["ร่วมกัน", "ด้วยกัน"]):
        if "83" not in combined_ids: combined_ids.append("83")
    if any(k in fact for k in ["พยายาม", "เกือบจะ"]):
        for s in ["80", "81"]:
            if s not in combined_ids: combined_ids.append(s)
 
    # --- 4. สร้าง context string ---
    context_str = "\n\n".join([
        f"[ID: {sid}] มาตรา {sid}: {master_law_dict[sid]}"
        for sid in list(set(combined_ids))
        if sid in master_law_dict
    ])
 
    # --- 5. Build prompt ตรงๆ (ไม่ผ่าน template แล้วค่อย replace) ---
    usr_p = f"""วินิจฉัยคดีดังต่อไปนี้:
 
=== ข้อเท็จจริง (Fact) ===
{fact}
 
=== ตัวบทกฎหมายอ้างอิง (Reference) ===
{context_str}
 
ตอบในรูปแบบ JSON:
{{
  "case_id": "{case_id}",
  "intent_and_act": "วิเคราะห์พฤติกรรมภายนอกบ่งบอกเจตนาภายในอย่างไร",
  "differential_analysis": "เปรียบเทียบมาตราที่ใกล้เคียงและเหตุผลที่เลือก",
  "criminal_charge": "ชื่อฐานความผิดหลักทางการ",
  "sections": ["เลขมาตราหลัก", "มาตราสนับสนุน"],
  "confidence_score": 0-100,
  "final_reasoning": "สรุปหลักนิติศาสตร์สั้นๆ"
}}"""
 
    raw      = call_typhoon(SYSTEM_PROMPT, usr_p)
    llm_dict = parse_llm_json(raw)
 
    # --- 6. Force Choice Retry ถ้า sections=[] (Merged TEST-B) ---
    if not llm_dict or not llm_dict.get("sections"):
        print(f"   ⚠️  {case_id} sections=[] → Force Choice retry (temp=0.3)")
        force_usr = usr_p + "\n\n**คำสั่งพิเศษ: ต้องเลือกมาตราอย่างน้อย 1 ข้อ ห้ามตอบลิสต์ว่างเด็ดขาด**"
        for attempt in range(2):
            try:
                response = typhoon_client.chat.completions.create(
                    model=TYPHOON_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": force_usr},
                    ],
                    max_tokens=MAX_TOKENS,
                    temperature=0.3,
                    response_format={"type": "json_object"},
                )
                llm_dict = parse_llm_json(response.choices[0].message.content)
                if llm_dict and llm_dict.get("sections"):
                    break
            except Exception as e:
                print(f"   ❌ Force retry attempt {attempt+1} failed: {e}")
            time.sleep(1)
 
    if not llm_dict:
        return {'case_id': case_id, 'error': 'LLM Failure after force retry'}
 
    return post_process_llm_output(
        llm_output=llm_dict,
        legal_db=list_of_all_legal_sections,
        context_section_ids=combined_ids,
    )
 
 
print(f"✅ [Step 7] Enhanced Pipeline พร้อม | Model: {TYPHOON_MODEL}")
print(f"   Features: Dual Retrieval + Fallback + Force Choice Retry")
 
# Smoke Test
if not df_judgement.empty:
    _result = analyze_criminal_liability(df_judgement.iloc[0]["fact"], "TEST-SMOKE")
    print(f"   Confidence: {_result.get('adjusted_confidence')}% | Sections: {_result.get('sections')}")

✅ [Step 7] Enhanced Pipeline พร้อม | Model: typhoon-v2.5-30b-a3b-instruct
   Features: Dual Retrieval + Fallback + Force Choice Retry
   Confidence: 88.0% | Sections: ['334', '288']


## 8

In [8]:
class LLMOutput(BaseModel):
    """Pydantic Schema ตรงกับ fields ที่ Step 6 สั่ง AI ตอบ"""
    case_id:               str
    intent_and_act:        str
    differential_analysis: str
    criminal_charge:       str
    sections:              list[str]
    confidence_score:      float = Field(ge=0, le=100)
    final_reasoning:       str

    @field_validator('sections')
    @classmethod
    def sections_not_empty(cls, v):
        if len(v) == 0: raise ValueError('sections ต้องมีอย่างน้อย 1 มาตรา')
        return v


def validate_llm_output(llm_dict: dict) -> tuple:
    """ตรวจสอบ JSON structure ด้วย Pydantic"""
    try:
        parsed = LLMOutput(**llm_dict)
        return True, parsed, ''
    except Exception as e:
        return False, None, str(e)


def is_valid_json_output(result: dict) -> bool:
    """ตรวจว่า result มี field หลักครบ"""
    required = ['case_id', 'verdict_mode', 'adjusted_confidence']
    return all(k in result for k in required) and 'error' not in result


def compute_f1_sections(pred_sections: list, gold_sections: list) -> dict:
    """คำนวณ Precision, Recall, F1 ระดับมาตรา"""
    pred_set = set(str(s) for s in pred_sections)
    gold_set = set(str(s) for s in gold_sections)
    if not pred_set and not gold_set: return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
    if not pred_set or not gold_set:  return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    tp        = len(pred_set & gold_set)
    precision = tp / len(pred_set)
    recall    = tp / len(gold_set)
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {'precision': round(precision, 4), 'recall': round(recall, 4), 'f1': round(f1, 4)}


print("✅ [Step 8] Validation Layer พร้อม (Schema + Metrics)")

✅ [Step 8] Validation Layer พร้อม (Schema + Metrics)


## 9

In [9]:
import concurrent.futures

# 1. ฟังก์ชัน Worker สำหรับรันรายคดีใน Dev Set
def run_dev_case(args):
    i, row = args
    case_id = f"DEV-{i:04d}"
    fact = str(row["fact"])
    gold_secs = row["clean_sections"]
    
    result = analyze_criminal_liability(fact, case_id, top_k=25) 
    
    result.update({
        "gold_sections": gold_secs,
        "fact": fact
    })
    
    if is_valid_json_output(result):
        pred_secs = result.get("sections", [])
        metrics = compute_f1_sections(pred_secs, gold_secs)
        result.update(metrics)
    else:
        result.update({"precision": 0.0, "recall": 0.0, "f1": 0.0})
        
    return result

# 2. เริ่มการประมวลผล Parallel
print(f"⏳ เริ่มประเมิน Dev Set {len(df_dev)} คดี แบบ Parallel (10 Workers)...")
start_dev = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    dev_results = list(tqdm(executor.map(run_dev_case, df_dev.iterrows()), total=len(df_dev)))

end_dev = time.time()

# 3. สรุปผลลัพธ์
valid_dev = [r for r in dev_results if is_valid_json_output(r)]
avg_f1_dev = sum(r["f1"] for r in valid_dev) / max(len(valid_dev), 1)
avg_prec_dev = sum(r["precision"] for r in valid_dev) / max(len(valid_dev), 1)
avg_rec_dev = sum(r["recall"] for r in valid_dev) / max(len(valid_dev), 1)

print("\n" + "=" * 55)
print("📊 PARALLEL DEV SET RESULTS")
print("=" * 55)
print(f"  Total Time    : {end_dev - start_dev:.2f} วินาที")
print(f"  Total Cases   : {len(dev_results)}")
print(f"  JSON Validity : {len(valid_dev)/len(dev_results):.1%} ({len(valid_dev)}/{len(dev_results)})")
print(f"  Avg Precision : {avg_prec_dev:.4f}")
print(f"  Avg Recall    : {avg_rec_dev:.4f}")
print(f"  Avg F1 Score  : {avg_f1_dev:.4f}")
print("=" * 55)

# 4. บันทึกผลลงไฟล์
with open("/kaggle/working/dev_results.json", "w", encoding="utf-8") as f:
    json.dump(dev_results, f, ensure_ascii=False, indent=2)
print("💾 บันทึกผล Dev ที่: /kaggle/working/dev_results.json")

⏳ เริ่มประเมิน Dev Set 81 คดี แบบ Parallel (10 Workers)...


  0%|          | 0/81 [00:00<?, ?it/s]

   ⚠️  DEV-0002 sections=[] → Force Choice retry (temp=0.3)


  2%|▏         | 2/81 [00:10<05:53,  4.48s/it]

   ⚠️  DEV-0012 sections=[] → Force Choice retry (temp=0.3)


...

   ⚠️  DEV-0057 sections=[] → Force Choice retry (temp=0.3)


 95%|█████████▌| 77/81 [01:48<00:04,  1.06s/it]

   ⚠️  DEV-0079 sections=[] → Force Choice retry (temp=0.3)


100%|██████████| 81/81 [01:57<00:00,  1.45s/it]


📊 PARALLEL DEV SET RESULTS
  Total Time    : 117.12 วินาที
  Total Cases   : 81
  JSON Validity : 100.0% (81/81)
  Avg Precision : 0.4712
  Avg Recall    : 0.4202
  Avg F1 Score  : 0.4208
💾 บันทึกผล Dev ที่: /kaggle/working/dev_results.json


## 10 

In [10]:
def categorize_error(result: dict, gold_sections: list) -> str:
    if 'error' in result: return 'NO_OUTPUT'
    pred_secs = set(result.get('sections', []))
    gold_secs = set(str(s) for s in gold_sections)
    invalid   = set(result.get('invalid_sections_flag', []))
    if pred_secs == gold_secs:              return 'CORRECT'
    if invalid:                             return 'HALLUCINATION'
    if pred_secs and not (pred_secs & gold_secs): return 'WRONG_SECTION'
    if gold_secs - pred_secs:               return 'INCOMPLETE'
    return 'PARTIAL'


error_counts   = {'CORRECT':0,'WRONG_SECTION':0,'HALLUCINATION':0,'INCOMPLETE':0,'PARTIAL':0,'NO_OUTPUT':0}
error_examples = {k: [] for k in error_counts}

for result in dev_results:
    gold = result.get('gold_sections', [])
    cat  = categorize_error(result, gold)
    error_counts[cat] += 1
    if cat != 'CORRECT' and len(error_examples[cat]) < 3:
        error_examples[cat].append({'case_id': result.get('case_id'), 'gold': gold,
                                    'predicted': result.get('sections', []), 'confidence': result.get('adjusted_confidence')})

total = len(dev_results)
print('=' * 55)
print('🔍 ERROR ANALYSIS')
print('=' * 55)
for cat, count in sorted(error_counts.items(), key=lambda x: -x[1]):
    bar = '█' * int(count / total * 30)
    print(f"  {cat:<15}: {count:>3} ({count/total:.1%}) {bar}")

with open('/kaggle/working/error_analysis.json', 'w', encoding='utf-8') as f:
    json.dump({'error_counts': error_counts, 'error_examples': error_examples}, f, ensure_ascii=False, indent=2)
print('\n✅ [Step 10] Error Analysis เสร็จสิ้น')

🔍 ERROR ANALYSIS
  INCOMPLETE     :  30 (37.0%) ███████████
  WRONG_SECTION  :  20 (24.7%) ███████
  HALLUCINATION  :  12 (14.8%) ████
  CORRECT        :  11 (13.6%) ████
  PARTIAL        :   8 (9.9%) ██
  NO_OUTPUT      :   0 (0.0%) 

✅ [Step 10] Error Analysis เสร็จสิ้น


## 11

In [11]:
def parse_lawids_for_cooc(lawids_str: str) -> list:
    if not isinstance(lawids_str, str) or not lawids_str.strip(): return []
    sections = []
    for token in lawids_str.split(','):
        m = re.match(r'CC-0*(\d+\S*)-\S+', token.strip())
        if m: sections.append(m.group(1))
    return sections


def build_cooccurrence(df, lawids_col: str = 'lawids') -> tuple:
    section_counts = defaultdict(int)
    pair_counts    = defaultdict(lambda: defaultdict(int))
    for lawids_str in df[lawids_col]:
        secs = list(set(parse_lawids_for_cooc(lawids_str)))
        for s in secs: section_counts[s] += 1
        for a, b in combinations(secs, 2):
            pair_counts[a][b] += 1
            pair_counts[b][a] += 1
    cooc = {}
    for a, partners in pair_counts.items():
        cooc[a] = {b: {'count': cnt, 'prob': round(cnt/section_counts[a], 3)} for b, cnt in partners.items()}
    return cooc, dict(section_counts)


DEPENDENT_SECTIONS = {'83','84','86','87','68','1','59','60','61'}


def build_companion_table(cooc: dict, section_counts: dict, threshold: float = 0.55) -> dict:
    table = {}
    for section, partners in cooc.items():
        companions = sorted([b for b, info in partners.items() if info['prob'] >= threshold],
                            key=lambda b: -partners[b]['prob'])
        table[section] = {'suggest_if_missing': companions, 'freq': section_counts.get(section, 0),
                          'top_pairs': {b: partners[b]['prob'] for b in companions}}
    return table


def apply_companion_check(result: dict, companion_table: dict, penalty_missing: int = 8, penalty_dangling: int = 5) -> dict:
    result = copy.deepcopy(result)
    if 'error' in result: return result
    pred_secs = {str(s) for s in result.get('sections', [])}
    warnings, suggestions, dangling = [], set(), set()
    for sec in pred_secs:
        if sec not in DEPENDENT_SECTIONS and sec in companion_table:
            for companion in companion_table[sec]['suggest_if_missing']:
                if companion not in pred_secs:
                    prob = companion_table[sec]['top_pairs'].get(companion, 0)
                    warnings.append(f'⚠️  ม.{sec} มักมากับ ม.{companion} ({prob:.0%}) แต่ไม่พบ')
                    suggestions.add(companion)
    non_dep = pred_secs - DEPENDENT_SECTIONS
    for sec in pred_secs:
        if sec in DEPENDENT_SECTIONS and not non_dep:
            dangling.add(sec)
            warnings.append(f'🔴 ม.{sec} ไม่มีมาตราหลัก')
    current_conf = result.get('adjusted_confidence') or 80
    if isinstance(current_conf, str): current_conf = int(re.sub(r'[^\d]', '', current_conf) or 80)
    penalty  = penalty_missing * min(len(suggestions), 3) + penalty_dangling * len(dangling)
    new_conf = max(0, int(current_conf) - penalty)
    result.update({'companion_warnings': warnings, 'companion_suggestions': list(suggestions),
                   'dangling_sections': list(dangling), 'adjusted_confidence': new_conf,
                   'possible_sections': list(set(result.get('possible_sections', [])) | suggestions - pred_secs)})
    return result


cooc, section_counts = build_cooccurrence(df_judgement, lawids_col='lawids')
COMPANION_TABLE = build_companion_table(cooc, section_counts, threshold=0.55)

dev_results_v2 = []
sug_count = dan_count = 0
for result in dev_results:
    r2 = apply_companion_check(result, COMPANION_TABLE)
    dev_results_v2.append(r2)
    if r2.get('companion_suggestions'): sug_count += 1
    if r2.get('dangling_sections'):     dan_count += 1

print(f'\n✅ [Step 11] Companion Table พร้อม')
print(f'   SUGGEST triggered: {sug_count}/{len(dev_results)} ({sug_count/len(dev_results):.0%})')
print(f'   PRUNE triggered  : {dan_count}/{len(dev_results)} ({dan_count/len(dev_results):.0%})')

with open('/kaggle/working/companion_table.json', 'w', encoding='utf-8') as f:
    json.dump(COMPANION_TABLE, f, ensure_ascii=False, indent=2)


✅ [Step 11] Companion Table พร้อม
   SUGGEST triggered: 15/81 (19%)
   PRUNE triggered  : 5/81 (6%)


## 12

In [13]:
def build_verify_prompt(fact: str, current_sections: list, candidate: str) -> tuple:
    current_str = ', '.join(f'ม.{s}' for s in current_sections)
    sys_p = """คุณคือผู้เชี่ยวชาญกฎหมายอาญาไทย ตอบเฉพาะ JSON เท่านั้น
รูปแบบ: {"add": true/false, "reason": "เหตุผลสั้นๆ"}
- add: true  = ข้อเท็จจริงรองรับมาตรานี้ชัดเจน
- add: false = ไม่รองรับ"""
    usr_p = f'ข้อเท็จจริง: {fact}\n\nมาตราที่ระบุแล้ว: {current_str}\nมาตราที่พิจารณาเพิ่ม: ม.{candidate}\n\nข้อเท็จจริงรองรับ ม.{candidate} หรือไม่?'
    return sys_p, usr_p



VERIFY_SLEEP = 1.0 
 
def sequential_verify(cases_to_verify: list, results_ref: list, index_ref: dict, label: str = '') -> tuple:
    """
    Companion Verify แบบ Sequential พร้อม sleep
    คืนค่า (results_v3, added_count, rejected_count)
    """
    results_v3    = copy.deepcopy(results_ref)
    added = rejected = 0
 
    for result in tqdm(cases_to_verify, desc=f"Verify {label}"):
        case_id     = result.get('case_id')
        fact        = result.get('fact', '')
        current_sec = result.get('sections', [])
        suggestions = result.get('companion_suggestions', [])
 
        if not fact and case_id:
            try:
                idx    = int(case_id.replace('DEV-', '').replace('TEST-', ''))
                src_df = df_dev if 'DEV-' in case_id else df_test
                if idx < len(src_df):
                    fact = str(src_df.iloc[idx]['fact'])
            except Exception:
                pass
 
        newly_added = []
        for candidate in suggestions:
            sys_p, usr_p = build_verify_prompt(fact, current_sec, candidate)
            raw    = call_typhoon(sys_p, usr_p)
            parsed = parse_llm_json(raw) if raw else None
            if parsed and parsed.get('add', False) and str(candidate) in list_of_all_legal_sections:
                newly_added.append(str(candidate))
                added += 1
            else:
                rejected += 1
            time.sleep(VERIFY_SLEEP)  
 
        if case_id in index_ref and newly_added:
            idx_r    = index_ref[case_id]
            combined = list(set(results_v3[idx_r].get('sections', [])) | set(newly_added))
            gold     = results_v3[idx_r].get('gold_sections', [])
            results_v3[idx_r]['sections']           = combined
            results_v3[idx_r]['verified_additions'] = newly_added
            results_v3[idx_r].update(compute_f1_sections(combined, gold))
 
    return results_v3, added, rejected
 
 
cases_to_verify = [r for r in dev_results_v2 if r.get('companion_suggestions') and 'error' not in r]
print(f"\n✅ [Step 12] Companion Verify: {len(cases_to_verify)} คดี (Sequential @ sleep={VERIFY_SLEEP}s)")
 
v3_index = {r.get('case_id'): i for i, r in enumerate(dev_results_v2)}
dev_results_v3, verified_added, verified_rejected = sequential_verify(
    cases_to_verify, dev_results_v2, v3_index, label='DEV'
)
 
print(f"   เพิ่มมาตรา: {verified_added} | ปฏิเสธ: {verified_rejected}")
 
f1_v2 = [compute_f1_sections(r.get('sections',[]), r.get('gold_sections',[]))['f1'] for r in dev_results_v2]
f1_v3 = [r.get('f1', 0.0) for r in dev_results_v3]
print(f"   Dev F1 After Verify: {sum(f1_v3)/len(f1_v3):.4f} (Before: {sum(f1_v2)/len(f1_v2):.4f})")
 
with open('/kaggle/working/dev_results_v3.json', 'w', encoding='utf-8') as f:
    json.dump(dev_results_v3, f, ensure_ascii=False, indent=2)
print('✅ [Step 12] Companion Verify เสร็จสิ้น')


✅ [Step 12] Companion Verify: 15 คดี (Sequential @ sleep=1.0s)


Verify DEV: 100%|██████████| 15/15 [00:28<00:00,  1.92s/it]

   เพิ่มมาตรา: 12 | ปฏิเสธ: 5
   Dev F1 After Verify: 0.4310 (Before: 0.4208)
✅ [Step 12] Companion Verify เสร็จสิ้น


## 13

In [25]:
import threading
import concurrent.futures


CHECKPOINT_PATH  = '/kaggle/working/test_checkpoint.json'
CHECKPOINT_EVERY = 50
MAX_WORKERS      = 15  
_lock            = threading.Lock() 

# --- [1] Logic การกู้คืนอัตโนมัติ (Recovery Logic) ---
def analyze_recovery_v1(fact: str, case_id: str, gold_secs: list) -> tuple:
    """
    Recovery pipeline — รันเมื่อ main pipeline ล้มเหลว
    ลองทีละ stage จนกว่าจะได้ sections
    คืนค่า (result, f1_score, stage_name)
    """
    stages = [
        ("K50_T02", {"top_k": 50, "temperature": 0.2}),
        ("K50_T06", {"top_k": 50, "temperature": 0.6}),
        ("K25_T05", {"top_k": 25, "temperature": 0.5}),
    ]
    for stage_name, params in stages:
        try:
            temp     = params.pop("temperature", 0.3)
            top_k    = params.get("top_k", 25)
            retrieved = hybrid_retrieve(fact, top_k=top_k)
            sys_p, usr_p = build_prompt(fact, retrieved, case_id)
            usr_p += "\n\n**คำสั่งพิเศษ: ต้องระบุมาตราอย่างน้อย 1 ข้อ ห้ามตอบ sections:[] เด็ดขาด**"
            response = typhoon_client.chat.completions.create(
                model=TYPHOON_MODEL,
                messages=[
                    {"role": "system", "content": sys_p},
                    {"role": "user",   "content": usr_p},
                ],
                max_tokens=MAX_TOKENS,
                temperature=temp,
                response_format={"type": "json_object"},
            )
            llm_dict = parse_llm_json(response.choices[0].message.content)
            if llm_dict and llm_dict.get("sections"):
                result = post_process_llm_output(
                    llm_output=llm_dict,
                    legal_db=list_of_all_legal_sections,
                    context_section_ids=retrieved["context_section_ids"],
                )
                f1 = compute_f1_sections(result.get("sections", []), gold_secs)["f1"]
                return result, f1, stage_name
        except Exception as e:
            print(f"   ❌ Recovery {stage_name} failed: {e}")
        time.sleep(1)

    return {"case_id": case_id, "sections": [], "error": "Recovery Failed"}, 0.0, "FAILED"

def analyze_with_auto_recovery(fact, case_id, gold_secs):
    """
    Pipeline หลัก: ลองรันปกติ (K=25) ถ้าล้มเหลวให้เข้าลูปกู้คืนทันที (K=50, Temp=0.6)[cite: 2]
    """
    # รอบที่ 1: รันแบบปกติ (Fast Track)[cite: 2]
    result = analyze_criminal_liability(fact, case_id, top_k=25) 
    
    # รอบที่ 2: ตรวจสอบปัญหา (Empty Sections หรือ LLM Error)[cite: 2]
    if not result.get("sections") or "error" in result:
        print(f"   🆘 {case_id} พบปัญหา (Empty/Error) -> กำลังเริ่ม Sequential Recovery...")
    
        recovered_res, new_f1, stage = analyze_recovery_v1(fact, case_id, gold_secs)
        
        if recovered_res.get("sections"):
            result = recovered_res
            result['recovery_log'] = f"Recovered by {stage}"
            print(f"   ✅ {case_id} กู้คืนสำเร็จ! (F1: {new_f1:.4f})")
            
    return result

# --- [2] Worker สำหรับการรันแบบ Parallel ---

def _run_one_case(args):
    """Worker ที่เรียกใช้ระบบ Auto-Recovery[cite: 2]"""
    i, row    = args
    case_id   = f'TEST-{i:04d}'
    fact      = str(row['fact'])
    gold_secs = row['clean_sections']
    
    # เรียกผ่านระบบกู้คืนอัตโนมัติ
    result = analyze_with_auto_recovery(fact, case_id, gold_secs)
    
    # แนบข้อมูลพื้นฐานกลับไปเพื่อใช้คำนวณ Metrics[cite: 2]
    result['gold_sections'] = gold_secs
    result['fact']          = fact
    
    if is_valid_json_output(result):
        result.update(compute_f1_sections(result.get('sections', []), gold_secs))
    else:
        result.update({'precision': 0.0, 'recall': 0.0, 'f1': 0.0})
        
    time.sleep(SLEEP_BETWEEN)
    return result

# --- [3] ส่วนควบคุมการรัน (Main Execution) ---

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        test_results = json.load(f)
    start_idx = len(test_results)
    print(f'♻️ Resume จาก checkpoint: {start_idx}/{len(df_test)} คดี')
else:
    test_results = []
    start_idx    = 0
    print('🆕 เริ่มใหม่ตั้งแต่ต้น')

print(f'\n⚠️ เริ่มรัน Full Run 965 คดี (ระบบ Auto-Recovery เปิดใช้งาน)\n')

remaining_rows = [(i, row) for i, row in df_test.iterrows() if i >= start_idx]
start_time = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_run_one_case, args): args[0] for args in remaining_rows}
    
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc='Test Eval'):
        result = future.result()
        with _lock:
            test_results.append(result)
            if len(test_results) % CHECKPOINT_EVERY == 0:
                with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
                    json.dump(test_results, f, ensure_ascii=False)
                print(f'💾 Checkpoint: {len(test_results)}/{len(df_test)} คดี')

elapsed = time.time() - start_time
print(f'\n✅ รัน pipeline ครบ {len(test_results)} คดี | เวลา: {elapsed:.1f}s')

# --- [4] Companion Check & Final Statistics ---[cite: 2]

print('\n⏳ เริ่มขั้นตอนสุดท้าย: Apply Companion Check & Verify...')
test_results_v2 = [apply_companion_check(r, COMPANION_TABLE) for r in test_results]
cases_tv        = [r for r in test_results_v2 if r.get('companion_suggestions') and 'error' not in r]

tv3_index = {r.get('case_id'): i for i, r in enumerate(test_results_v2)}
test_results_v3, tv_added, tv_rejected = sequential_verify(
    cases_tv, test_results_v2, tv3_index, label='TEST'
)

valid_results = [r for r in test_results_v3 if 'error' not in r and 'f1' in r]
n_total       = len(test_results_v3)
n_valid       = len(valid_results)
avg_f1        = sum(r['f1'] for r in valid_results) / n_valid if n_valid else 0
json_validity = (n_valid / n_total) * 100

print('\n' + '=' * 55)
print('🏁 FINAL TEST SET RESULTS (AUTO-RECOVERY ENABLED)')
print('=' * 55)
print(f"  Total Cases   : {n_total}")
print(f"  JSON Validity : {json_validity:.1f}% ({n_valid}/{n_total})")
print(f"  Avg F1 Score  : {avg_f1:.4f}")
print(f"  Total Time    : {elapsed:.1f}s")
print('=' * 55)

output = {
    'timestamp': datetime.now().isoformat(),
    'results': test_results_v3,
    'summary': {'avg_f1': round(avg_f1, 4), 'validity': round(json_validity, 2)}
}
with open('/kaggle/working/test_results_final.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print('🗑️ ลบ checkpoint แล้ว')

print('💾 บันทึกผลสมบูรณ์ที่: /kaggle/working/test_results_final.json')

🆕 เริ่มใหม่ตั้งแต่ต้น

⚠️ เริ่มรัน Full Run 965 คดี (ระบบ Auto-Recovery เปิดใช้งาน)



Test Eval:   0%|          | 4/965 [00:13<36:58,  2.31s/it]  

   ⚠️  TEST-0001 sections=[] → Force Choice retry (temp=0.3)


Test Eval:   2%|▏         | 23/965 [00:28<06:26,  2.44it/s]

   ⚠️  TEST-0013 sections=[] → Force Choice retry (temp=0.3)


...

   ⚠️  TEST-0957 sections=[] → Force Choice retry (temp=0.3)
   ⚠️  TEST-0959 sections=[] → Force Choice retry (temp=0.3)


Test Eval: 100%|██████████| 965/965 [17:56<00:00,  1.12s/it]



✅ รัน pipeline ครบ 965 คดี | เวลา: 1076.4s

⏳ เริ่มขั้นตอนสุดท้าย: Apply Companion Check & Verify...


Verify TEST: 100%|██████████| 153/153 [05:51<00:00,  2.30s/it]


🏁 FINAL TEST SET RESULTS (AUTO-RECOVERY ENABLED)
  Total Cases   : 965
  JSON Validity : 100.0% (965/965)
  Avg F1 Score  : 0.5068
  Total Time    : 1076.4s
🗑️ ลบ checkpoint แล้ว
💾 บันทึกผลสมบูรณ์ที่: /kaggle/working/test_results_final.json


## 14 re-cheak

In [26]:
# Appendix A: ดู System Errors และ Zero-F1
with open('/kaggle/working/test_results_final.json', 'r', encoding='utf-8') as f:
    final_data = json.load(f)

all_results   = final_data['results']
system_errors = [r for r in all_results if 'error' in r]
zero_f1_cases = [r for r in all_results if 'error' not in r and r.get('f1') == 0]
empty_pred    = [r for r in all_results if not r.get('sections') and r.get('gold_sections')]

print('📊 สรุปภาพรวมปัญหา:')
print(f"   System Errors (LLM Failure) : {len(system_errors)} คดี")
print(f"   Zero F1 (ตอบผิดทั้งหมด)   : {len(zero_f1_cases)} คดี")
print(f"   Empty Prediction (ไม่ตอบ)  : {len(empty_pred)} คดี")
print('=' * 50)

for i, case in enumerate(system_errors[:5]):
    print(f"❌ [{i+1}] {case.get('case_id')} | สาเหตุ: {case.get('error')}")

if zero_f1_cases:
    print('\n💡 ตัวอย่าง Zero F1 (3 คดี):')
    for case in random.sample(zero_f1_cases, min(3, len(zero_f1_cases))):
        print(f"📌 {case.get('case_id')} | Gold: {case.get('gold_sections')} | Pred: {case.get('sections')}")

📊 สรุปภาพรวมปัญหา:
   System Errors (LLM Failure) : 0 คดี
   Zero F1 (ตอบผิดทั้งหมด)   : 257 คดี
   Empty Prediction (ไม่ตอบ)  : 0 คดี

💡 ตัวอย่าง Zero F1 (3 คดี):
📌 TEST-0261 | Gold: ['335'] | Pred: ['352']
📌 TEST-0327 | Gold: ['326'] | Pred: ['328']
📌 TEST-0957 | Gold: ['295'] | Pred: ['334', '344']


In [27]:
# Appendix B: Retry คดีที่ timeout / System Error
target_ids  = [r['case_id'] for r in system_errors]
test_subset = df_test[df_test.index.map(lambda i: f'TEST-{i:04d}').isin(target_ids)]
print(f"🚀 Retry {len(test_subset)} คดีที่ fail")

retry_results = []
for i, row in tqdm(test_subset.iterrows(), total=len(test_subset)):
    case_id = f'TEST-{i:04d}'
    try:
        result = analyze_criminal_liability(str(row['fact']), case_id, top_k=25)
        result['gold_sections'] = row['clean_sections']
        result['fact']          = str(row['fact'])
        if 'error' not in result:
            result.update(compute_f1_sections(result.get('sections', []), row['clean_sections']))
        else:
            result.update({'precision': 0.0, 'recall': 0.0, 'f1': 0.0})
        retry_results.append(result)
    except Exception as e:
        retry_results.append({'case_id': case_id, 'error': f'Still failing: {str(e)}'})
    time.sleep(0.1)

for r in retry_results:
    if 'f1' in r and 'error' not in r:
        print(f"✅ {r['case_id']}: F1={r['f1']:.4f} | Pred={r['sections']}")
    else:
        print(f"❌ {r['case_id']}: {r.get('error')}")

🚀 Retry 0 คดีที่ fail


0it [00:00, ?it/s]
